# pathlib: Working with File Paths

Use `pathlib` to build, inspect, and create file paths. The examples here use a temporary folder, so nothing in your project changes.

## 1. Build paths with `/`

In [ ]:
from pathlib import Path

base = Path("data")
raw_csv = base / "raw" / "insurance.csv"

print(raw_csv)
print(type(raw_csv))

`/` joins path pieces. It works on Windows and Linux, so you don't need to write separators yourself.

## 2. Where am I?

Relative paths are resolved from the **current working directory**, which is where Python was started. In a notebook, that's the folder the notebook is in.

In [ ]:
import os
from pathlib import Path

print("working directory:", Path.cwd())
print("file exists?", Path("data/raw/insurance.csv").exists())

This is the problem you hit in `scratch.ipynb`. From inside `notebooks/`, `data/raw/insurance.csv` points to `notebooks/data/raw/insurance.csv`, which doesn't exist.

Two fixes:

- Use a path relative to the notebook: `Path("../data/raw/insurance.csv")`. This works only while the notebook stays in `notebooks/`.
- Build the path from a fixed reference point, which the next section shows.

## 3. Find the project root from a fixed point

In a script, `__file__` gives the script's own path. Notebooks don't have `__file__`, so from a notebook, go up from the current folder until you find a marker, such as `pyproject.toml`:

In [ ]:
from pathlib import Path

def find_project_root(start: Path = Path.cwd()) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("no pyproject.toml found above " + str(start))

ROOT = find_project_root()
print("project root:", ROOT)
print("csv path:    ", ROOT / "data" / "raw" / "insurance.csv")

In a script, `Path(__file__).resolve().parents[1]` gives the project root, because `parents` counts up from the file's folder. Here's a quick look at how `parents` works:

In [ ]:
example = Path("/projects/health-claim-cost-model/src/load_to_sqlite.py")

print("parent:    ", example.parent)
print("parents[0]:", example.parents[0])
print("parents[1]:", example.parents[1])

## 4. Split a path into parts

In [ ]:
p = Path("data/processed/insurance.db")

print("name:  ", p.name)      # insurance.db
print("stem:  ", p.stem)      # insurance
print("suffix:", p.suffix)    # .db
print("parts: ", p.parts)

## 5. Create folders and files (in a temporary location)

In [ ]:
import tempfile

tmp = Path(tempfile.mkdtemp())

folder = tmp / "data" / "processed"
folder.mkdir(parents=True, exist_ok=True)

print("created:", folder.exists())

`parents=True` creates any missing folders above it. `exist_ok=True` means running it again doesn't raise an error. Use both together, since it's the usual way to make sure a folder exists before writing to it.

Write and read text:

In [ ]:
note = folder / "notes.txt"
note.write_text("hello from pathlib\n")

print(note.read_text())
print("size in bytes:", note.stat().st_size)

## 6. Find files with `glob`

In [ ]:
for name in ["a.csv", "b.csv", "c.txt"]:
    (folder / name).write_text("x")

print(sorted(p.name for p in folder.glob("*.csv")))
print(sorted(p.name for p in folder.glob("*")))

`glob("*.csv")` matches only in that folder. Use `rglob("*.csv")` to search subfolders too.

## 7. Clean up

The temporary folder is ours, so remove it when you're done:

In [ ]:
import shutil

shutil.rmtree(tmp)
print("removed:", not tmp.exists())

## Exercises

**Exercise 1.** Given `Path("data/raw/insurance.csv")`, print its folder, file name, and extension.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
p = Path("data/raw/insurance.csv")
print(p.parent)
print(p.name)
print(p.suffix)
```

</details>

**Exercise 2.** Use `find_project_root()` to build the path to `data/processed/insurance.db`, then check whether the file exists.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
db = find_project_root() / "data" / "processed" / "insurance.db"
print(db, db.exists())
```

</details>

## Summary

- `/` joins path parts safely.
- Relative paths depend on the working directory, which is why `scratch.ipynb` failed.
- `parents[n]` counts up from a file, and `find_project_root()` counts up until it finds a marker.
- `mkdir(parents=True, exist_ok=True)` makes sure a folder exists.
- `glob` finds files by pattern.